In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)


In [4]:
from langchain_core.messages import HumanMessage
response=model.invoke([HumanMessage("what is the capital of india")])

In [5]:
response

AIMessage(content='The capital of India is New Delhi.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 41, 'total_tokens': 50, 'completion_time': 0.007138923, 'completion_tokens_details': None, 'prompt_time': 0.002166458, 'prompt_tokens_details': None, 'queue_time': 0.053046171, 'total_time': 0.009305381}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d80c3-3728-7ab0-9375-c7ecac817665-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 9, 'total_tokens': 50})

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()

    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [8]:
config={"configurable":{"session_id":"user1"}}
with_message_history.invoke([HumanMessage(content='what is a dog')],config=config)

AIMessage(content="A dog is a domesticated mammal that belongs to the family Canidae. Dogs are closely related to wolves and are known for their loyalty, intelligence, and ability to form strong bonds with humans.\n\nPhysical Characteristics:\n\n* Dogs come in a wide range of shapes and sizes, from small breeds like the Chihuahua to large breeds like the Great Dane.\n* They have a furry coat, a tail, and four legs.\n* Some breeds have erect ears, while others have floppy ears.\n* Dogs have a keen sense of smell and hearing.\n\nBehavior:\n\n* Dogs are social animals that thrive on interaction with their human family and other dogs.\n* They are known for their loyalty and affection towards their owners.\n* Dogs are highly trainable and can be trained to perform a variety of tasks, from simple obedience commands to complex tasks like search and rescue.\n* Some breeds are naturally more energetic and require more exercise than others.\n\nTypes of Dogs:\n\n* Sporting dogs (e.g. Golden Retri

prompts templates

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant answers all the questions to the best of your ability."),
        MessagesPlaceholder(variable_name="input")
    ]
)

chain=prompt|model

In [13]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input"
)

In [14]:
with_message_history.invoke(
    {"input": [HumanMessage(content="hi my name is abhi")]},
    config=config
)

AIMessage(content="Hello Abhi, it's nice to meet you. I'm happy to chat with you about anything you'd like. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 433, 'total_tokens': 466, 'completion_time': 0.126596319, 'completion_tokens_details': None, 'prompt_time': 0.050833385, 'prompt_tokens_details': None, 'queue_time': 0.121043666, 'total_time': 0.177429704}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d80d9-7620-79a2-aa68-c6329cbd2231-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 433, 'output_tokens': 33, 'total_tokens': 466})

In [15]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

In [18]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

chain=(
    RunnablePassthrough.assign(input=itemgetter("input")|trimmer)|prompt|model
)



In [20]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input"
)
messages = [
    SystemMessage(content="You are helpful"),
    HumanMessage(content="Hi"),
    AIMessage(content="Hello!"),
    HumanMessage(content="Tell me a joke")
]




config={"configurable": {"session_id": "user1"}}
with_message_history.invoke({"input": messages}, config=config)
with_message_history.invoke({"input": "what was the last joke?"}, config=config)

c:\Users\HP\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_core\language_models\base.py:336: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


AIMessage(content='The last joke I told you was: \n\nWhat do you call a fake noodle?\n\nAn impasta!\n\nDid you find it amusing?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 111, 'total_tokens': 140, 'completion_time': 0.090042547, 'completion_tokens_details': None, 'prompt_time': 0.007955786, 'prompt_tokens_details': None, 'queue_time': 0.051112344, 'total_time': 0.097998333}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d80e4-05e1-7eb0-94c6-d945170d1d65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 29, 'total_tokens': 140})